In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy import signal

# 設置顯示所有列
pd.set_option('display.max_columns', None)
# 設置顯示所有行
pd.set_option('display.max_rows', None)
# 設置每列的最大寬度
pd.set_option('display.max_colwidth', None)
# 禁用列省略號
pd.set_option('display.expand_frame_repr', False)


In [ ]:
time = np.linspace(0, 100e-6, 100000)

control_pulse_amplitude = 0.01 # V
control_pulse_start_time = 10e-9 # second

In [ ]:
data = {
    "Time": time,
    "Signal": control_pulse_amplitude * np.heaviside(time - control_pulse_start_time, 1),
    "Step Response of Bias Tee": (np.exp(-time / 41e-6)) * np.heaviside(time - control_pulse_start_time, 1),
}

data["Impulse Response of Bias Tee"] = np.gradient(data["Step Response of Bias Tee"], (data["Time"][1] - data["Time"][0]))
data["signal Out"] = np.convolve(data["Signal"], data["Impulse Response of Bias Tee"], mode="same")[:len(data["Time"])] * (data["Time"][1] - data["Time"][0])
df = pd.DataFrame(data)

In [ ]:
# 使用 Plotly 繪製信號圖表
fig = go.Figure()

# 繪製 Signal
fig.add_trace(go.Scatter(x=df['Time'], y=df['Signal'], mode='lines', name='Signal In'))

# 繪製 Step Response of Bias Tee
fig.add_trace(go.Scatter(x=df['Time'], y=df['Step Response of Bias Tee'], mode='lines', name='Step Response of Bias Tee'))

# 繪製 Impulse Response
fig.add_trace(go.Scatter(x=df['Time'], y=df['Impulse Response of Bias Tee'], mode='lines', name='Impulse Response of Bias Tee'))

# 繪製 Signal Out
fig.add_trace(go.Scatter(x=df['Time'], y = df['signal Out'], mode='lines', name = 'Signal Out'))

# 設定圖表標題和軸標籤
fig.update_layout(
    title='Signal, Step Response of Bias Tee and Signal Out over Time',
    xaxis_title='Time',
    yaxis_title='Amplitude'
)

# 顯示圖表
fig.show()


In [ ]:
df.head()
print(df['Impulse Response'][0])

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# 定義時間變量和控制脈衝參數
time_resolution = 1e-9  # 使用更小的時間步長
time = np.arange(0, 10e-6, time_resolution)  # 定義時間範圍

control_pulse_amplitude = 1
control_pulse_start_time = 2e-6

# 定義階躍響應 s(t)
step_response = (1 - np.exp(-time / 41e-6)) * np.heaviside(time - control_pulse_start_time, 1)

# 計算脈衝響應 h(t) 作為階躍響應 s(t) 的導數
impulse_response = np.gradient(step_response, time_resolution)

# 確保在 t = 0 時脈衝響應為零
impulse_response[0] = 0

# 檢查脈衝響應的正確性
print("Impulse Response at the beginning:", impulse_response[:10])

# 定義輸入信號 x(t)
signal = control_pulse_amplitude * np.heaviside(time - control_pulse_start_time, 1)

# 確保在 t = 0 時輸入信號為零
signal[0] = 0

# 檢查輸入信號的正確性
print("Signal at the beginning:", signal[:10])

# 添加零填充來處理卷積邊界效應
signal_padded = np.pad(signal, (len(impulse_response)//2,), mode='constant')
impulse_response_padded = np.pad(impulse_response, (len(signal)//2,), mode='constant')

# 計算輸出信號 y(t) 作為 x(t) 與 h(t) 的卷積
signal_out_same = np.convolve(signal_padded, impulse_response_padded, mode="same") * time_resolution
signal_out_full = np.convolve(signal_padded, impulse_response_padded, mode="full") * time_resolution
signal_out_valid = np.convolve(signal_padded, impulse_response_padded, mode="valid") * time_resolution

# 檢查卷積結果的正確性
print("Signal Out (same) at the beginning:", signal_out_same[:10])
print("Signal Out (full) at the beginning:", signal_out_full[:10])
print("Signal Out (valid) at the beginning:", signal_out_valid[:10])

# 創建 DataFrame
data = {
    "Time": time,
    "Signal": signal,
    "Step Response": step_response,
    "Impulse Response": impulse_response,
    "Signal Out (same)": signal_out_same[:len(time)],  # 確保長度與時間相同
    "Signal Out (full)": signal_out_full[:len(time)],  # 確保長度與時間相同
    "Signal Out (valid)": np.pad(signal_out_valid, (0, len(time) - len(signal_out_valid)), 'constant')  # 填充到相同長度
}
df = pd.DataFrame(data)

# 使用 Plotly 繪製圖表
fig = go.Figure()

# 繪製 Signal
fig.add_trace(go.Scatter(x=df['Time'], y=df['Signal'], mode='lines', name='Signal'))

# 繪製 Step Response
fig.add_trace(go.Scatter(x=df['Time'], y=df['Step Response'], mode='lines', name='Step Response'))

# 繪製 Impulse Response
fig.add_trace(go.Scatter(x=df['Time'], y=df['Impulse Response'], mode='lines', name='Impulse Response'))

# 繪製 Signal Out (same)
fig.add_trace(go.Scatter(x=df['Time'], y=df['Signal Out (same)'], mode='lines', name='Signal Out (same)'))

# 繪製 Signal Out (full)
fig.add_trace(go.Scatter(x=df['Time'], y=df['Signal Out (full)'], mode='lines', name='Signal Out (full)'))

# 繪製 Signal Out (valid)
fig.add_trace(go.Scatter(x=df['Time'], y=df['Signal Out (valid)'], mode='lines', name='Signal Out (valid)'))

# 設定圖表標題和軸標籤
fig.update_layout(
    title='Signal, Step Response, Impulse Response, and Signal Out over Time',
    xaxis_title='Time (s)',
    yaxis_title='Amplitude'
)

# 顯示圖表
fig.show()
